# Lista 5 – Questão 07: Regras de Associação com Apriori (base Titanic)
**Disciplina:** Inteligência Artificial – Profa. Cristiane Neri Nobre

Notebook adaptado a partir do código `Apriori.ipynb` disponibilizado na disciplina (biblioteca `apyori`).
Objetivo: gerar regras de associação que descrevam o perfil dos passageiros e sua relação com a sobrevivência,
usando itens de **presença** (ex.: `Classe=1a`) e de **ausência** (ex.: `NÃO Classe=1a`).

In [ ]:
!pip install apyori

In [2]:
import itertools
import pandas as pd
from apyori import apriori

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

## 1. Leitura da base

In [3]:
# No Colab, faça upload do arquivo titanic_completo.csv (ou ajuste o caminho)
base = pd.read_csv('titanic_completo.csv')
print(base.shape)
base.head()

(1309, 14)


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.00,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [4]:
# Quantidade de valores ausentes por atributo
base.isna().sum()

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

## 1(a). Tratamento de valores ausentes e remoção de atributos

* **Idade (263 ausentes):** imputada pela **mediana da idade do grupo do título** extraído do nome (Mr, Mrs, Miss, Master, Outro).
  O título carrega informação de idade (ex.: *Master* = menino), por isso é melhor do que usar a mediana geral.
* **Tarifa (1 ausente):** mediana da tarifa da classe do passageiro.
* **Embarque (2 ausentes):** moda (Southampton).
* **Atributos removidos:** `name` (identificador; usado apenas para extrair o título), `ticket` (identificador),
  `cabin` (77% ausente), `home.dest` (43% ausente e alta cardinalidade), `boat` e `body`
  (**vazamento de informação**: só existe bote para quem sobreviveu e número do corpo para quem morreu – gerariam regras óbvias).
* `sibsp` e `parch` foram combinados em um único atributo **tamanho da família**.

In [5]:
# Extrai o título do nome (texto entre a vírgula e o ponto)
base['Titulo'] = base['name'].str.extract(r',\s*([^\.]+)\.')[0].str.strip()
# Agrupa títulos raros/equivalentes
base['Titulo'] = base['Titulo'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
                                         'Lady': 'Mrs', 'the Countess': 'Mrs', 'Dona': 'Mrs'})
base.loc[~base['Titulo'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Titulo'] = 'Outro'

# Mediana da idade por título -> usada para preencher as idades ausentes
mediana_idade = base.groupby('Titulo')['age'].median()
print(mediana_idade)
base['age'] = base['age'].fillna(base['Titulo'].map(mediana_idade))

# Tarifa ausente -> mediana da classe ; Embarque ausente -> moda
base['fare'] = base['fare'].fillna(base.groupby('pclass')['fare'].transform('median'))
base['embarked'] = base['embarked'].fillna(base['embarked'].mode()[0])

# Tamanho da família = irmãos/cônjuge + pais/filhos + o próprio passageiro
base['familia'] = base['sibsp'] + base['parch'] + 1

# Remove atributos que não interessam às regras
base = base.drop(columns=['name', 'ticket', 'cabin', 'boat', 'body', 'home.dest', 'sibsp', 'parch', 'Titulo'])
print(base.isna().sum().sum(), 'valores ausentes restantes')

Titulo
Master     4.0
Miss      22.0
Mr        29.0
Mrs       35.5
Outro     49.0
Name: age, dtype: float64
0 valores ausentes restantes


## 1(b). Discretização dos atributos numéricos

* **Idade:** `0-12` (crianças – protocolo "mulheres e crianças primeiro"), `13-17` (adolescentes/menores de idade),
  `18-30` (jovens adultos, maior grupo), `31-50` (adultos), `51+` (mais velhos).
* **Tarifa (libras):** os quantis de 33% e 66% são ≈ 8,5 e 26; usamos os cortes arredondados **10** e **30**, que dividem a base em
  três grupos de tamanho parecido e coincidem com o preço típico de cada classe (veja a mediana por classe abaixo).
* **Família:** `Sozinho` (1), `Pequena(2-4)`, `Grande(5+)` – famílias grandes tinham mais dificuldade de se manter juntas na evacuação.

In [6]:
print(base['fare'].quantile([0.33, 0.66]))
print(base.groupby('pclass')['fare'].median())

0.33     8.5167
0.66    26.0000
Name: fare, dtype: float64
pclass
1    60.0000
2    15.0458
3     8.0500
Name: fare, dtype: float64


In [7]:
d = pd.DataFrame()
d['Classe']     = base['pclass'].map({1: '1a', 2: '2a', 3: '3a'})
d['Sexo']       = base['sex'].map({'male': 'Masculino', 'female': 'Feminino'})
d['Idade']      = pd.cut(base['age'], [-1, 12, 17, 30, 50, 200],
                         labels=['0-12', '13-17', '18-30', '31-50', '51+']).astype(str)
d['Tarifa']     = pd.cut(base['fare'], [-1, 10, 30, 1e6],
                         labels=['Baixa(<=10)', 'Média(10-30)', 'Alta(>30)']).astype(str)
d['Família']    = pd.cut(base['familia'], [0, 1, 4, 100],
                         labels=['Sozinho', 'Pequena(2-4)', 'Grande(5+)']).astype(str)
d['Embarque']   = base['embarked'].map({'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'})
d['Sobreviveu'] = base['survived'].map({1: 'Sim', 0: 'Não'})

for c in d.columns:
    print(c, d[c].value_counts().to_dict())

Classe {'3a': 709, '1a': 323, '2a': 277}
Sexo {'Masculino': 843, 'Feminino': 466}
Idade {'18-30': 682, '31-50': 370, '0-12': 102, '51+': 95, '13-17': 60}
Tarifa {'Baixa(<=10)': 492, 'Média(10-30)': 474, 'Alta(>30)': 343}
Família {'Sozinho': 790, 'Pequena(2-4)': 437, 'Grande(5+)': 82}
Embarque {'Southampton': 916, 'Cherbourg': 270, 'Queenstown': 123}
Sobreviveu {'Não': 809, 'Sim': 500}


## 1(c) e 2. Codificação binária (one-hot) com itens de presença e de ausência

1. `pd.get_dummies` cria uma coluna para cada par **atributo=valor** (item de presença).
2. Para cada atributo com **3 ou mais valores** (Classe, Idade, Tarifa, Família, Embarque) criamos o item de ausência
   **`NÃO atributo=valor`**, que é simplesmente a negação lógica (`~`) da coluna de presença.
3. Para atributos **binários** (Sexo e Sobreviveu) não criamos ausência, pois `NÃO Sexo=Masculino` é idêntico a `Sexo=Feminino`
   (seria só um item duplicado).
4. Cada passageiro vira uma transação contendo os nomes das colunas verdadeiras: 7 itens de presença + 12 itens de ausência = 19 itens.

In [8]:
presenca = pd.get_dummies(d, prefix_sep='=')        # itens de presença

atrib_multivalor = [c for c in d.columns if d[c].nunique() > 2]
cols_aus = [c for c in presenca.columns if c.split('=')[0] in atrib_multivalor]
ausencia = (~presenca[cols_aus]).add_prefix('NÃO ')   # itens de ausência (negação)

binaria = pd.concat([presenca, ausencia], axis=1).astype(bool)
print(binaria.shape)
binaria.head()

(1309, 38)


,Classe=1a,Classe=2a,Classe=3a,Sexo=Feminino,Sexo=Masculino,Idade=0-12,Idade=13-17,Idade=18-30,Idade=31-50,Idade=51+,...,NÃO Idade=51+,NÃO Tarifa=Alta(>30),NÃO Tarifa=Baixa(<=10),NÃO Tarifa=Média(10-30),NÃO Família=Grande(5+),NÃO Família=Pequena(2-4),NÃO Família=Sozinho,NÃO Embarque=Cherbourg,NÃO Embarque=Queenstown,NÃO Embarque=Southampton
0,True,False,False,True,False,False,False,True,False,False,...,True,False,True,True,True,True,False,True,True,False
1,True,False,False,False,True,True,False,False,False,False,...,True,False,True,True,True,False,True,True,True,False
2,True,False,False,True,False,True,False,False,False,False,...,True,False,True,True,True,False,True,True,True,False
3,True,False,False,False,True,False,False,True,False,False,...,True,False,True,True,True,False,True,True,True,False
4,True,False,False,True,False,False,False,True,False,False,...,True,False,True,True,True,False,True,True,True,False


In [9]:
# Transforma a matriz binária em lista de transações (formato exigido pelo apyori)
transacoes = [list(binaria.columns[linha]) for linha in binaria.values]
print(len(transacoes), 'transações')
transacoes[0]

1309 transações


['Classe=1a',
 'Sexo=Feminino',
 'Idade=18-30',
 'Tarifa=Alta(>30)',
 'Família=Sozinho',
 'Embarque=Southampton',
 'Sobreviveu=Sim',
 'NÃO Classe=2a',
 'NÃO Classe=3a',
 'NÃO Idade=0-12',
 'NÃO Idade=13-17',
 'NÃO Idade=31-50',
 'NÃO Idade=51+',
 'NÃO Tarifa=Baixa(<=10)',
 'NÃO Tarifa=Média(10-30)',
 'NÃO Família=Grande(5+)',
 'NÃO Família=Pequena(2-4)',
 'NÃO Embarque=Cherbourg',
 'NÃO Embarque=Queenstown']

## 3. Geração das regras

Funções auxiliares:
* `gerar_regras`: executa o Apriori e monta o DataFrame (como no código original da disciplina) com suporte, confiança e lift.
* `trivial`: marca regras **logicamente óbvias** criadas pela codificação de ausência, por exemplo
  `{NÃO Embarque=Southampton, NÃO Embarque=Queenstown} → {Embarque=Cherbourg}` (mesmo atributo no antecedente e no consequente)
  ou `{Classe=1a, NÃO Classe=3a}` (mesmo atributo repetido – a ausência é implicada pela presença).
* `nao_redundante`: remove uma regra se algum subconjunto do antecedente produz quase a mesma confiança (melhoria < 0,02),
  ou seja, o item extra não acrescenta informação.

In [10]:
def atributo(item):
    return (item[4:] if item.startswith('NÃO ') else item).split('=')[0]

def trivial(a, b):
    aa, bb = [atributo(i) for i in a], [atributo(i) for i in b]
    return len(set(aa)) < len(aa) or len(set(bb)) < len(bb) or bool(set(aa) & set(bb))

def gerar_regras(sup, conf, max_len=4):
    linhas = []
    for resultado in apriori(transacoes, min_support=sup, min_confidence=conf, max_length=max_len):
        for r in resultado.ordered_statistics:
            a, b = list(r.items_base), list(r.items_add)
            if len(a) == 0 or len(b) == 0:
                continue
            linhas.append((a, b, resultado.support, r.confidence, r.lift, trivial(a, b)))
    return pd.DataFrame(linhas, columns=['Antecedente', 'Consequente', 'suporte', 'confianca', 'lift', 'trivial'])

def confianca(a, b):
    m = binaria[list(a)].all(axis=1)
    return (m & binaria[list(b)].all(axis=1)).sum() / m.sum()

def nao_redundante(R, min_melhoria=0.02):
    manter = []
    for _, r in R.iterrows():
        a = r.Antecedente
        ok = all(r.confianca - confianca(s, r.Consequente) >= min_melhoria
                 for k in range(1, len(a)) for s in itertools.combinations(a, k))
        manter.append(ok)
    return R[manter]

def filtrar(R):
    # remove triviais, mantém consequente com 1 item e lift >= 1,2 (associação de fato)
    return R[(~R.trivial) & (R.Consequente.str.len() == 1) & (R.lift >= 1.2)]

eh_sobrev = lambda x: len(x) == 1 and x[0].startswith('Sobreviveu')

## 3(b). Teste de diferentes suportes e confianças

`max_length=4` (antecedente com até 3 itens) para manter as regras interpretáveis.

In [11]:
tabela = []
for sup, conf in itertools.product([0.03, 0.05, 0.10, 0.20], [0.6, 0.7, 0.8, 0.9]):
    R = gerar_regras(sup, conf)
    U = filtrar(R)
    S = U[U.Consequente.apply(eh_sobrev)]
    N = nao_redundante(S) if len(S) else S
    tabela.append({'suporte_min': sup, 'confianca_min': conf,
                   'regras_apriori': len(R),
                   'apos_filtro': len(U),
                   'conseq_sobrev': len(S),
                   'sobrev_nao_redund': len(N),
                   'Sim': int(sum(N.Consequente.str[0] == 'Sobreviveu=Sim')) if len(N) else 0,
                   'Não': int(sum(N.Consequente.str[0] == 'Sobreviveu=Não')) if len(N) else 0})
tabela = pd.DataFrame(tabela)
tabela.to_csv('grade_parametros.csv', index=False)
tabela

,suporte_min,confianca_min,regras_apriori,apos_filtro,conseq_sobrev,sobrev_nao_redund,Sim,Não
0,0.03,0.6,123966,10493,1273,249,157,92
1,0.03,0.7,91183,8119,952,187,95,92
2,0.03,0.8,65199,5838,446,102,41,61
3,0.03,0.9,44039,3602,82,25,19,6
4,0.05,0.6,108868,8356,1041,192,117,75
5,0.05,0.7,79577,6403,770,144,69,75
6,0.05,0.8,56539,4520,336,79,35,44
7,0.05,0.9,37638,2744,51,16,15,1
8,0.10,0.6,79205,4769,652,99,43,56
9,0.10,0.7,56579,3725,536,77,21,56


**Escolha:** suporte mínimo = **0,10** (pelo menos 131 passageiros por regra) e confiança mínima = **0,80**.
Com suportes de 0,03/0,05 surgem dezenas de milhares de regras, muitas apoiadas em grupos pequenos;
com 0,20 quase não sobram regras para *Sobreviveu=Sim* (apenas 38% sobreviveram e os grupos de alta sobrevivência são menores);
confiança 0,90 deixa pouquíssimas regras. A configuração 0,10/0,80 gera regras robustas e em número analisável.

In [12]:
SUP, CONF = 0.10, 0.80
regras = gerar_regras(SUP, CONF)
print('Regras geradas pelo Apriori:', len(regras))
uteis = filtrar(regras)
print('Após remover triviais / lift < 1,2:', len(uteis))

Regras geradas pelo Apriori: 38633
Após remover triviais / lift < 1,2: 2532


## 4(a). Regras mais relevantes (todas) ordenadas por lift

In [13]:
gerais = nao_redundante(uteis.sort_values('lift', ascending=False).head(400))
top_gerais = gerais.head(15)[['Antecedente', 'Consequente', 'suporte', 'confianca', 'lift']]
top_gerais.to_pickle('top_gerais.pkl')
top_gerais

,Antecedente,Consequente,suporte,confianca,lift
407,"[NÃO Família=Sozinho, Classe=1a]",[Tarifa=Alta(>30)],0.120703,0.969325,3.699261
259,"[Classe=1a, Família=Pequena(2-4)]",[Tarifa=Alta(>30)],0.112299,0.967105,3.690789
6224,"[NÃO Idade=0-12, Tarifa=Alta(>30), Sobreviveu=Sim]",[Classe=1a],0.122995,0.889503,3.604827
6042,"[NÃO Família=Grande(5+), NÃO Idade=18-30, Tarifa=Alta(>30)]",[Classe=1a],0.127578,0.865285,3.506681
4649,"[NÃO Idade=0-12, Tarifa=Alta(>30), Família=Pequena(2-4)]",[Classe=1a],0.108480,0.855422,3.466709
6005,"[NÃO Idade=0-12, NÃO Família=Grande(5+), Tarifa=Alta(>30)]",[Classe=1a],0.177998,0.850365,3.446216
451,"[Tarifa=Alta(>30), Sobreviveu=Sim]",[Classe=1a],0.126050,0.841837,3.411654
398,"[NÃO Família=Grande(5+), Tarifa=Alta(>30)]",[Classe=1a],0.181818,0.826389,3.349050
260,"[Tarifa=Alta(>30), Família=Pequena(2-4)]",[Classe=1a],0.112299,0.816667,3.309649
450,"[Sobreviveu=Sim, Classe=1a]",[Tarifa=Alta(>30)],0.126050,0.825000,3.148469


## 4(b). Regras cujo consequente é a sobrevivência

In [14]:
sobrev = nao_redundante(uteis[uteis.Consequente.apply(eh_sobrev)])
sim = sobrev[sobrev.Consequente.str[0] == 'Sobreviveu=Sim'].sort_values('lift', ascending=False)
nao = sobrev[sobrev.Consequente.str[0] == 'Sobreviveu=Não'].sort_values('lift', ascending=False)
sim.to_pickle('sim.pkl'); nao.to_pickle('nao.pkl')
print(len(sim), 'regras -> Sobreviveu=Sim')
sim[['Antecedente', 'Consequente', 'suporte', 'confianca', 'lift']]

13 regras -> Sobreviveu=Sim


,Antecedente,Consequente,suporte,confianca,lift
31617,"[Sexo=Feminino, NÃO Tarifa=Média(10-30), NÃO Classe=3a]",[Sobreviveu=Sim],0.110008,0.972973,2.547243
31631,"[Tarifa=Alta(>30), Sexo=Feminino, NÃO Classe=3a]",[Sobreviveu=Sim],0.110008,0.972973,2.547243
449,"[Classe=1a, Sexo=Feminino]",[Sobreviveu=Sim],0.106188,0.965278,2.527097
36671,"[NÃO Família=Grande(5+), Tarifa=Alta(>30), Sexo=Feminino]",[Sobreviveu=Sim],0.104660,0.964789,2.525817
2950,"[Sexo=Feminino, NÃO Classe=3a]",[Sobreviveu=Sim],0.177998,0.932000,2.439976
37944,"[NÃO Idade=0-12, Tarifa=Alta(>30), Sexo=Feminino]",[Sobreviveu=Sim],0.106952,0.903226,2.364645
4272,"[Tarifa=Alta(>30), Sexo=Feminino]",[Sobreviveu=Sim],0.111536,0.869048,2.275167
36624,"[NÃO Família=Grande(5+), NÃO Tarifa=Baixa(<=10), Sexo=Feminino]",[Sobreviveu=Sim],0.206264,0.841121,2.202056
36359,"[NÃO Família=Grande(5+), NÃO Idade=18-30, Sexo=Feminino]",[Sobreviveu=Sim],0.142857,0.827434,2.166221
18909,"[NÃO Tarifa=Baixa(<=10), Sexo=Feminino, Família=Pequena(2-4)]",[Sobreviveu=Sim],0.138273,0.822727,2.153900


In [15]:
print(len(nao), 'regras -> Sobreviveu=Não')
nao[['Antecedente', 'Consequente', 'suporte', 'confianca', 'lift']]

28 regras -> Sobreviveu=Não


,Antecedente,Consequente,suporte,confianca,lift
8733,"[NÃO Idade=0-12, Classe=2a, Sexo=Masculino]",[Sobreviveu=Não],0.111536,0.912500,1.476468
27673,"[NÃO Idade=0-12, NÃO Classe=1a, Sexo=Masculino]",[Sobreviveu=Não],0.409473,0.878689,1.421759
27404,"[NÃO Classe=1a, NÃO Família=Pequena(2-4), Sexo=Masculino]",[Sobreviveu=Não],0.349885,0.872381,1.411553
4273,"[Sexo=Masculino, Tarifa=Baixa(<=10)]",[Sobreviveu=Não],0.255157,0.865285,1.400072
14281,"[Idade=18-30, Embarque=Southampton, Sexo=Masculino]",[Sobreviveu=Não],0.222307,0.863501,1.397186
32702,"[NÃO Idade=0-12, NÃO Embarque=Cherbourg, Sexo=Masculino]",[Sobreviveu=Não],0.418640,0.857590,1.387621
37870,"[NÃO Idade=0-12, NÃO Tarifa=Alta(>30), Sexo=Masculino]",[Sobreviveu=Não],0.411765,0.856916,1.386530
16747,"[NÃO Idade=0-12, Embarque=Southampton, Sexo=Masculino]",[Sobreviveu=Não],0.379679,0.855422,1.384112
716,"[Classe=2a, Sexo=Masculino]",[Sobreviveu=Não],0.111536,0.853801,1.381490
2553,"[NÃO Classe=1a, Sexo=Masculino]",[Sobreviveu=Não],0.430863,0.849398,1.374365


## Taxas de sobrevivência de referência (para interpretar o lift)

In [16]:
print('Taxa geral:', round((d.Sobreviveu == 'Sim').mean(), 3))
for c in ['Sexo', 'Classe', 'Idade']:
    print(pd.crosstab(d[c], d.Sobreviveu, normalize='index').round(3), '\n')
# Efeito da ausência: homens adultos x todos os homens
h = d.Sexo == 'Masculino'
print('Morte - todos os homens:', round((d[h].Sobreviveu == 'Não').mean(), 3))
print('Morte - homens com NÃO Idade=0-12:', round((d[h & (d.Idade != '0-12')].Sobreviveu == 'Não').mean(), 3))
print('Morte - meninos 0-12:', round((d[h & (d.Idade == '0-12')].Sobreviveu == 'Não').mean(), 3))

Taxa geral: 0.382
Sobreviveu    Não    Sim
Sexo                    
Feminino    0.273  0.727
Masculino   0.809  0.191 

Sobreviveu    Não    Sim
Classe                  
1a          0.381  0.619
2a          0.570  0.430
3a          0.745  0.255 

Sobreviveu    Não    Sim
Idade                   
0-12        0.441  0.559
13-17       0.550  0.450
18-30       0.677  0.323
31-50       0.573  0.427
51+         0.600  0.400 

Morte - todos os homens: 0.809
Morte - homens com NÃO Idade=0-12: 0.833
Morte - meninos 0-12: 0.483
